<a href="https://colab.research.google.com/github/AndresMontesDeOca/IIA/blob/main/3-TP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de búsqueda
**Facundo A. Lucianna - Inteligencia Artificial - CEIA - FIUBA**

Un algoritmo de búsqueda toma un problema de búsqueda como entrada y retorna una solución, o una indicación de falla.

La idea es buscar un camino que llegue al estado objetivo. Para ello, vamos a construir un árbol que irá avanzando por los estados del grafo hasta llegar al estado objetivo.

![arbol_de_hanoi](https://github.com/AndresMontesDeOca/IIA/blob/main/img/tree_hanoi.png?raw=1)

Cada nodo del árbol corresponde a un **estado** y las aristas corresponden a una **acción**. Es importante destacar que el árbol **NO** es el grafo de estados. El grafo describe todo el conjunto de estados y las acciones que permiten pasar de un estado a otro. El árbol describe el camino entre estos estados para alcanzar el objetivo.

Para poder aplicar los algoritmos, debemos definir las estructuras de datos necesarias para hacer seguimiento del árbol.

## Nodos del árbol

Los nodos del árbol están representados por los siguientes componentes:

- **State**: El estado del espacio de estados que corresponde al nodo.
- **Node Parent**: El nodo en el árbol de búsqueda que generó este nodo.
- **Action**: La acción que se aplicó al nodo padre para generar este nodo.
- **Path-Cost**: El costo de un camino desde el nodo inicial hasta este nodo.

Definamos el **Problema** de la Torre de Hanoi:

In [1]:
import sys
import os

# Add the correct path to 'aima_libs' within 'intro_ia'
repo_path = 'intro_ia/clase2/exercise'
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print(f"Added {repo_path} to sys.path")

# Clone the repository containing the desired aima_libs package
!git clone https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/intro_ia.git

Added intro_ia/clase2/exercise to sys.path
Cloning into 'intro_ia'...
remote: Enumerating objects: 2884, done.
remote: Counting objects: 100% (228/228), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 2884 (delta 77), reused 200 (delta 61), pack-reused 2656 (from 2)
Receiving objects: 100% (2884/2884), 626.76 MiB | 23.10 MiB/s, done.
Resolving deltas: 100% (1122/1122), done.
Updating files: 100% (310/310), done.


In [2]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi

initial_state = StatesHanoi([5, 4, 3, 2, 1], [], [], max_disks=5)
goal_state = StatesHanoi([], [], [5, 4, 3, 2, 1], max_disks=5)

problem = ProblemHanoi(initial=initial_state, goal=goal_state)

Empezamos con la primera estructura, que implementamos con la clase `NodeHanoi`. Esta clase tiene implementados los siguientes:

**Atributos:**
* `state`: El estado que el nodo contiene. Representa un estado particular de ubicación de los discos.
* `parent`: Es el nodo padre de este nodo. Si el nodo es la raíz, es `None`.
* `action`: La acción que se aplicó al nodo padre para llegar a este nodo. Si es la raíz, es `None`.
* `path_cost`: El costo del camino desde la raíz del árbol hasta este nodo.
* `depth`: La profundidad del árbol en la que se encuentra el nodo. Si es la raíz, es cero; si es un hijo de la raíz, es igual a 1, y así sucesivamente.

**Métodos:**
* `child_node()`: Genera un nodo hijo a partir de una acción.
* `expand()`: Expande la frontera de este nodo, devolviendo los nodos hijos al aplicar `expand`.
* `solution()`: Retorna en una lista la secuencia de acciones que van desde la raíz hasta este nodo.
* `path()`: Retorna una lista de nodos que van desde la raíz hasta este nodo.
* `generate_solution_for_simulator()`: Este método permite obtener una salida para simular con PyGame. En otro notebook profundizaremos más en este aspecto.

Además, tiene implementados métodos que nos permiten hacer diferentes operaciones en Python:

* Podemos comparar dos nodos para ver si son iguales (haciendo `node1 == node2`)
* Podemos preguntar si un nodo es mayor que otro (haciendo `node1 > node2`), lo que significa si el costo acumulado de un nodo es mayor que otro.
* Tenemos una representación en cadena del estado, por lo que cuando hacemos `print()` se observa el estado dentro del nodo con el texto `<Node >`.
* También podemos obtener un hash del estado con `hash(estado)`


In [3]:
from aima_libs.tree_hanoi import NodeHanoi

In [4]:
# Definimos la raíz del árbol
root = NodeHanoi(state=initial_state)

In [5]:
print("El arbol tiene como raíz a:")
print(root)

El arbol tiene como raíz a:
<Node HanoiState: 5 4 3 2 1 |  | >


Desde un nodo y con el problema definido, podemos encontrar la frontera, que es la separación entre el grafo que ya ha sido explorado por el algoritmo de búsqueda y aquel que aún no ha sido explorado.

![frontera_en_arbol_de_hanoi](https://github.com/AndresMontesDeOca/IIA/blob/main/img/tree_hanoi_frontier.png?raw=1)

### Expandir la frontera del nodo raíz

Expandimos la frontera del nodo raíz:


In [6]:
lista_nodos_fronteras = root.expand(problem=problem)

In [7]:
for nodos in lista_nodos_fronteras:
    print(nodos)

<Node HanoiState: 5 4 3 2 | 1 | >
<Node HanoiState: 5 4 3 2 |  | 1>


Los estados que corresponden a estos nuevos nodos son:

![state](https://github.com/AndresMontesDeOca/IIA/blob/main/img/state_hanoi5.png?raw=1)

Por lo tanto, el árbol para este problema se verá así:

![tree](https://github.com/AndresMontesDeOca/IIA/blob/main/img/tree_hanoi2.png?raw=1)

Es decir, se generan dos nodos nuevos con los siguientes estados:

- El disco verde (el más pequeño) se mueve a la varilla del medio.
- El disco verde (el más pequeño) se mueve a la varilla derecha.

Quedémonos con el nodo cuyo estado tiene el disco verde en la varilla del medio:

In [8]:
next_node = lista_nodos_fronteras[0]

Ahora, veamos su estado:

In [9]:
next_node.state

HanoiState: 5 4 3 2 | 1 | 

Vemos quién es el padre, que debería ser la raíz:

In [10]:
next_node.parent

<Node HanoiState: 5 4 3 2 1 |  | >

El costo acumulado desde la raíz hasta este nodo es:

In [11]:
next_node.path_cost

1.0

Y la profundidad en la que nos encontramos en el árbol:

In [12]:
next_node.depth

1

### Expandir la frontera del nodo siguiente


Expandimos ahora la frontera de este nodo:

In [13]:
lista_nodos_fronteras2 = next_node.expand(problem=problem)

In [14]:
for nodos in lista_nodos_fronteras2:
    print(nodos)

<Node HanoiState: 5 4 3 | 1 | 2>
<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 |  | 1>


Veamos cómo quedó ahora el árbol:

![tree](https://github.com/AndresMontesDeOca/IIA/blob/main/img/tree_hanoi3.png?raw=1)

Observamos lo siguiente: se generaron tres nuevos nodos en la frontera. De estos, hay dos que llaman la atención.

- Hay un nodo cuyo estado es igual al del nodo padre. Esto ocurre porque, si movemos el disco verde de vuelta a la varilla de la izquierda, *"retornamos"* al estado inicial.
- Hay un nodo que tiene el mismo estado que el segundo nodo obtenido del padre.

Esto es importante de destacar: **múltiples nodos pueden tener el mismo estado**, pero el costo para llegar a ese nodo y la secuencia de acciones desde la raíz hasta ese nodo serán diferentes.

Veamos ahora el nodo cuyo estado no está repetido:

In [15]:
next_node2 = lista_nodos_fronteras2[0]

Vemos su estado:

In [16]:
next_node2.state

HanoiState: 5 4 3 | 1 | 2

Vemos quién es el padre:

In [17]:
next_node2.parent

<Node HanoiState: 5 4 3 2 | 1 | >

El costo acumulado desde la raíz hasta este nodo es:

In [18]:
next_node2.path_cost

2.0

Y la profundidad en la que estamos en el árbol:

In [19]:
next_node2.depth

2

Veamos el camino desde la raíz hasta este nodo:

In [20]:
for nodos in next_node2.path():
    print(nodos)

<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 | 1 | >
<Node HanoiState: 5 4 3 | 1 | 2>


Y las acciones que se aplicaron desde el inicio hasta este nodo:

In [21]:
for nodos in next_node2.solution():
    print(nodos)

Move disk 1 from 1 to 2
Move disk 2 from 1 to 3


## Colas

La pregunta es: ¿cómo hacemos para elegir cuál nodo seleccionar para explorar la frontera? Para ello necesitamos una estructura de datos que nos permita explorar la frontera. Esta estructura es vital para el algoritmo de búsqueda, ya que nos permite seleccionar qué nodo vamos a expandir primero. Como vimos en los videos, elegir el tipo de estructura para expandir es lo que define el tipo de algoritmo.

La frontera se expande usando **colas**, y tenemos tres tipos:

- Una cola **FIFO** (Primero entra, primero sale) que toma los nodos en el mismo orden en que se agregan.
- Una cola **LIFO** (Último en entrar, primero en salir, o "stack") que quita el nodo más reciente.
- Una **cola prioritaria** que quita primero los nodos con el mínimo costo, de acuerdo con una función de evaluación `f()`.

### FIFO

Veamos una implementación de cola FIFO usando una lista:

In [22]:
# aaaaa

In [23]:
fifo = []
print(lista_nodos_fronteras[0])

<Node HanoiState: 5 4 3 2 | 1 | >


La implementación debe incorporar las siguientes funciones:

- **Add(Frontier)**: Inserta el nodo en su correspondiente lugar de la cola. En el caso de la FIFO, inserta los nodos en el orden en que van llegando, utilizando el método `insert()`.

In [24]:
fifo.insert(0, lista_nodos_fronteras[0])
fifo.insert(0, lista_nodos_fronteras[1])

for nodos in fifo:
    print(nodos)

fifo

<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 2 | 1 | >


[<Node HanoiState: 5 4 3 2 |  | 1>, <Node HanoiState: 5 4 3 2 | 1 | >]

- **Is-empty(frontier)**: Retorna `True` si no hay nodos en la frontera. En el caso de la implementación con una lista, podemos preguntar si la cantidad de elementos es cero.

In [25]:
print("La cola está vacia?")
if len(fifo) == 0:
    print("La cola esta vacía")
else:
    print("La cola tiene elementos")

La cola está vacia?
La cola tiene elementos


- **Pop(frontier)**: Quita el primer nodo en la cola. Con las listas, podemos usar el método `pop()`.

In [26]:
new_node = fifo.pop()
print(f"El nodo que sacamos es {new_node}")


El nodo que sacamos es <Node HanoiState: 5 4 3 2 | 1 | >


Podemos ver que, una vez sacado el nodo, el mismo ya no está en la fila, y que particularmente sacamos el primer nodo que entró:

In [27]:
print("Los nodos que quedan en la fila son:")
print(fifo)

Los nodos que quedan en la fila son:
[<Node HanoiState: 5 4 3 2 |  | 1>]


Si expandimos la frontera del nuevo nodo que tenemos, podemos agregarlo a la cola:

In [28]:
print(lista_nodos_fronteras2[0])

<Node HanoiState: 5 4 3 | 1 | 2>


In [29]:
lista_nodos_fronteras2 = new_node.expand(problem=problem)

# Insertamos los nodos de frontera en el orden que nos fue presentado:
fifo.insert(0, lista_nodos_fronteras2[0])
fifo.insert(0, lista_nodos_fronteras2[1])
fifo.insert(0, lista_nodos_fronteras2[2])

In [30]:
for nodos in fifo:
    print(nodos)

<Node HanoiState: 5 4 3 2 |  | 1>
<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 | 1 | 2>
<Node HanoiState: 5 4 3 2 |  | 1>


- **Top(frontier)**: Podemos ver cuál es nuestro siguiente nodo sin sacarlo:

In [31]:
print(f"El siguiente nodo que podemos sacar es {fifo[-1]}")

El siguiente nodo que podemos sacar es <Node HanoiState: 5 4 3 2 |  | 1>


Este es el segundo nodo que introdujimos cuando agregamos la frontera de la raíz. Como vemos, se respeta el principio *"Primero entra, primero sale"*.

### LIFO

Ahora veamos cómo podemos implementar un **stack**. Esto también lo podemos hacer con una lista.

In [32]:
lifo = []

- **Add(Frontier)**: Inserta el nodo en su correspondiente lugar de la cola. En el caso de la LIFO, se inserta de forma apilada para que el último nodo que se inserte, esté listo para salir. Esto lo hacemos usando `append()`:

In [33]:
print(lista_nodos_fronteras[0])

<Node HanoiState: 5 4 3 2 | 1 | >


In [34]:
lifo.append(lista_nodos_fronteras[0])
lifo.append(lista_nodos_fronteras[1])

for nodos in lifo:
    print(nodos)

<Node HanoiState: 5 4 3 2 | 1 | >
<Node HanoiState: 5 4 3 2 |  | 1>


In [35]:
lifo

[<Node HanoiState: 5 4 3 2 | 1 | >, <Node HanoiState: 5 4 3 2 |  | 1>]

> ⚠️ **Nota**: Los nodos están ordenados de forma inversa al caso de la fila FIFO.

- **Is-empty(frontier)**: Retorna `True` si no hay nodos en la frontera. En el caso de la implementación con una lista, podemos preguntar si la cantidad de elementos es cero.

In [36]:
print("El stack está vacio?")
if len(lifo) == 0:
    print("El stack esta vacio")
else:
    print("El stack tiene elementos")

El stack está vacio?
El stack tiene elementos


- **Pop(frontier)**: Quita el primer nodo en la cola. Con las listas, usamos el método `pop()`.

In [37]:
new_node = lifo.pop()
print(f"El nodo que sacamos es {new_node}")

El nodo que sacamos es <Node HanoiState: 5 4 3 2 |  | 1>


Podemos ver que, una vez sacado el nodo, el mismo ya no está en la fila, y que particularmente sacamos el último nodo que entró:

In [38]:
print("Los nodos que quedan en la fila son:")
print(lifo)

Los nodos que quedan en la fila son:
[<Node HanoiState: 5 4 3 2 | 1 | >]


Si expandimos la frontera del nuevo nodo que tenemos, podemos agregarlo a la cola:

In [39]:
lista_nodos_fronteras2 = new_node.expand(problem=problem)

lifo.append(lista_nodos_fronteras2[0])
lifo.append(lista_nodos_fronteras2[1])
lifo.append(lista_nodos_fronteras2[2])

In [40]:
for nodos in lifo:
    print(nodos)

<Node HanoiState: 5 4 3 2 | 1 | >
<Node HanoiState: 5 4 3 | 2 | 1>
<Node HanoiState: 5 4 3 2 1 |  | >
<Node HanoiState: 5 4 3 2 | 1 | >


- **Top(frontier)**: Podemos ver cuál es nuestro siguiente nodo sin sacarlo:

In [41]:
print(f"El siguiente nodo que podemos sacar es {lifo[-1]}")

El siguiente nodo que podemos sacar es <Node HanoiState: 5 4 3 2 | 1 | >


### Conclusión

Todo lo que hemos visto hasta ahora *—desde cómo definimos el problema hasta cómo se define la cola—* son ejemplos de implementación. Estos sirven como base, pero cada persona o proyecto es libre de usar estas soluciones o implementar las propias, de acuerdo a las necesidades del problema o del entorno en el que se esté trabajando.

----

## Búsqueda Primero en Anchura

Con todo lo que fuimos implementando, ahora podemos aplicar un algoritmo de búsqueda. Observá todo lo que tuvimos que definir e implementar previamente para poder llegar hasta este punto.

Vamos a implementar el algoritmo de **búsqueda primero en anchura (Breadth-First Search)**, tal como vimos en el video. Este algoritmo comienza desde la raíz y va expandiendo todos los nodos nivel por nivel utilizando una cola **FIFO**.

![breadth_first_search](https://github.com/AndresMontesDeOca/IIA/blob/main/img/breadth_first_search.png?raw=1)

### Probamos con menos discos

Empecemos con menos discos para probar cómo implementarlo y luego pasamos al caso con 5 discos:

In [42]:
# Inicializamos el problema
initial_state = StatesHanoi([3, 2, 1], [], [], max_disks=3)
goal_state = StatesHanoi([], [], [3, 2, 1], max_disks=3)
problem = ProblemHanoi(initial=initial_state, goal=goal_state)

frontier = [NodeHanoi(problem.initial)]  # Creamos una cola FIFO con el nodo inicial

find_solution = False

# Mientras que la cola no este vacia
while len(frontier) != 0:
    node = frontier.pop()  # Extraemos el primer nodo de la cola
    if problem.goal_test(node.state):  # Comprobamos si hemos alcanzado el estado objetivo
        last_node = node
        find_solution = True
        print("Encontramos la solución")
        break

    # Agregamos a la cola todos los nodos sucesores del nodo actual
    for next_node in node.expand(problem):
        frontier.insert(0, next_node)

if not find_solution:
    print("No se encontró la solución")

Encontramos la solución


Una vez encontrada, podemos analizar el último nodo del árbol y ver cuánto costó llegar a la solución:

In [43]:
print(f'Longitud del camino de la solución: {last_node.state.accumulated_cost}')
print(f'Profundidad del arbol donde se encontró la solución: {last_node.depth}')

Longitud del camino de la solución: 7.0
Profundidad del arbol donde se encontró la solución: 7


Como todos los pasos tienen el mismo costo, la **búsqueda primero en anchura** garantiza que el camino encontrado es el más corto posible en términos de número de acciones. Es decir, el algoritmo encontró la solución en la mínima profundidad del árbol.

Además, podemos analizar el desempeño del algoritmo:

Podemos visualizar el camino desde el estado inicial hasta la solución:

In [45]:
for nodos in last_node.path():
    print(nodos)

<Node HanoiState: 3 2 1 |  | >
<Node HanoiState: 3 2 |  | 1>
<Node HanoiState: 3 | 2 | 1>
<Node HanoiState: 3 | 2 1 | >
<Node HanoiState:  | 2 1 | 3>
<Node HanoiState: 1 | 2 | 3>
<Node HanoiState: 1 |  | 3 2>
<Node HanoiState:  |  | 3 2 1>


Y las acciones que el agente debería aplicar para alcanzar el estado objetivo:

In [46]:
for act in last_node.solution():
    print(act)

Move disk 1 from 1 to 3
Move disk 2 from 1 to 2
Move disk 1 from 3 to 2
Move disk 3 from 1 to 3
Move disk 1 from 2 to 1
Move disk 2 from 2 to 3
Move disk 1 from 1 to 3
